# Topic 41 — Experimentation & Reproducibility
### Theory → random seeds everywhere → a simple experiment log → results tables → MLflow/W&B preview.

A result you (or anyone else) can't REPRODUCE isn't very useful — especially for a paper. This
topic is about discipline: controlling randomness, and keeping organized records of what you ran,
with what settings, and what happened.

In [ ]:
import random
import numpy as np
import torch
import pandas as pd
import json
from datetime import datetime

## 1. Random seeds — everywhere, not just one place

Randomness sneaks in from multiple independent sources: Python's own `random`, NumPy, PyTorch's
CPU RNG, PyTorch's GPU RNG, and even `train_test_split`'s shuffling. Missing even ONE means your
results won't reproduce exactly.

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # For full determinism on GPU (slower, but exact reproducibility):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("seed set. random.random():", random.random())
print("np.random.rand():", np.random.rand())
print("torch.rand(1):", torch.rand(1).item())

# Re-running set_seed(42) resets everything back to the exact same starting point
set_seed(42)
print("\nafter resetting seed again:")
print("random.random():", random.random())   # should match the first run exactly

## 2. Reproducible train/test splits

`random_state` in sklearn's `train_test_split`/`KFold`/etc (Topic 6) IS a seed — always set it
explicitly, and use the SAME value across an experiment so comparisons between runs are fair
(different splits would make results incomparable, confusing "which config is better" with
"which split happened to be easier").

In [ ]:
from sklearn.model_selection import train_test_split

X_dummy = np.arange(20).reshape(-1, 1)
y_dummy = np.array([0,1]*10)

split1 = train_test_split(X_dummy, y_dummy, test_size=0.3, random_state=42)
split2 = train_test_split(X_dummy, y_dummy, test_size=0.3, random_state=42)   # same seed

print("split1 test indices match split2 test indices:",
      np.array_equal(split1[1], split2[1]))
# ALWAYS true with the same random_state -- this IS what "reproducible split" means.

## 3. Hyperparameter logging & a simple experiment log

Before reaching for a dedicated tool, even a plain list of dictionaries (or a CSV/DataFrame) that
records every experiment's settings and results is far better than nothing — and is exactly what
a paper's "experiments" table usually starts life as.

In [ ]:
experiment_log = []

def log_experiment(name, hyperparams, metrics):
    entry = {
        "experiment_name": name,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        **{f"param_{k}": v for k, v in hyperparams.items()},
        **{f"metric_{k}": v for k, v in metrics.items()},
    }
    experiment_log.append(entry)
    return entry

# Simulate a few experiment runs (in reality these come from actually training models)
log_experiment("logreg_baseline", {"C": 1.0, "ngram_range": (1,1)}, {"f1": 0.72, "precision": 0.75, "recall": 0.70})
log_experiment("logreg_bigrams", {"C": 1.0, "ngram_range": (1,2)}, {"f1": 0.78, "precision": 0.80, "recall": 0.76})
log_experiment("svm_tuned", {"C": 10.0, "ngram_range": (1,2)}, {"f1": 0.81, "precision": 0.83, "recall": 0.79})
log_experiment("lstm_v1", {"hidden_dim": 32, "embed_dim": 16}, {"f1": 0.83, "precision": 0.85, "recall": 0.81})

log_df = pd.DataFrame(experiment_log)
print(log_df)

In [ ]:
# Save/load the log as a file -- now it survives across Colab sessions
log_df.to_csv("experiment_log.csv", index=False)
loaded_log = pd.read_csv("experiment_log.csv")
print("reloaded log:\n", loaded_log[["experiment_name", "metric_f1"]])

## 4. Model checkpoints (previewed here, full treatment in Topic 42)

Save not just the final model, but the config that produced it, so any result can be traced back
and reproduced exactly.

In [ ]:
def save_checkpoint(model_state, hyperparams, metrics, path):
    checkpoint = {
        "hyperparams": hyperparams,
        "metrics": metrics,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }
    with open(path.replace(".pt", "_meta.json"), "w") as f:
        json.dump(checkpoint, f, indent=2)
    # torch.save(model_state, path)  -- the actual model weights, covered fully in Topic 42
    print(f"saved metadata to {path.replace('.pt', '_meta.json')}")

save_checkpoint(
    model_state=None,   # a real state_dict would go here
    hyperparams={"hidden_dim": 32, "lr": 0.01, "seed": 42},
    metrics={"f1": 0.83},
    path="lstm_v1.pt"
)

with open("lstm_v1_meta.json") as f:
    print(json.load(f))

## 5. Dataset versioning

If your dataset changes over time (cleaned differently, more examples added, relabeled), results
from different versions aren't directly comparable. Simplest approach: track a version tag/hash
alongside every experiment log entry.

In [ ]:
import hashlib

def dataset_fingerprint(df):
    # A simple content hash -- changes if ANY cell in the dataframe changes
    content_str = df.to_csv(index=False)
    return hashlib.md5(content_str.encode()).hexdigest()[:10]

dummy_dataset_v1 = pd.DataFrame({"text": ["a", "b", "c"], "label": [0, 1, 0]})
dummy_dataset_v2 = pd.DataFrame({"text": ["a", "b", "c", "d"], "label": [0, 1, 0, 1]})  # added a row

print("v1 fingerprint:", dataset_fingerprint(dummy_dataset_v1))
print("v2 fingerprint:", dataset_fingerprint(dummy_dataset_v2))
# Different fingerprints confirm these are meaningfully different dataset versions --
# log this fingerprint alongside each experiment so you always know which data version produced which result.

## 6. Building a clean results table (what actually goes in your paper)

In [ ]:
final_results = log_df[["experiment_name", "metric_f1", "metric_precision", "metric_recall"]].copy()
final_results.columns = ["Model", "F1", "Precision", "Recall"]
final_results = final_results.sort_values("F1", ascending=False).reset_index(drop=True)
print(final_results.round(3))

# This is close to publication-ready -- format numbers consistently, sort by your headline metric,
# and you have your paper's results table.

## 7. MLflow & Weights & Biases — when your project outgrows a CSV

Once you're running MANY experiments, dedicated tools automatically log hyperparameters, metrics,
and even model artifacts, with a searchable UI to compare runs — worth adopting once the plain-CSV
approach above starts feeling limiting, not necessarily on day one.

In [ ]:
# MLflow sketch (not run here -- pip install mlflow):
#
# import mlflow
# with mlflow.start_run(run_name="svm_tuned"):
#     mlflow.log_params({"C": 10.0, "ngram_range": "(1,2)"})
#     mlflow.log_metrics({"f1": 0.81, "precision": 0.83, "recall": 0.79})
#     mlflow.sklearn.log_model(model, "model")

# Weights & Biases sketch (not run here -- pip install wandb):
#
# import wandb
# wandb.init(project="cyberbullying-detection", config={"C": 10.0, "ngram_range": "(1,2)"})
# wandb.log({"f1": 0.81, "precision": 0.83, "recall": 0.79})

print("Both tools give you a web dashboard to compare runs -- consider adopting once you have")
print("10+ experiments and the plain CSV log starts getting hard to navigate.")

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Log 3 more (simulated) experiments to experiment_log with different hyperparameter combos.
# 2. Add a 'dataset_version' column to log_experiment using dataset_fingerprint() from part 5.
# 3. Write a function best_experiment(log_df, metric='metric_f1') that returns the single best row.
# 4. Set up set_seed(42) at the very top of your REAL project notebook (once you have one) so
#    every run of it is reproducible from here forward.

---
### Next up: **Topic 42 — Model Saving** (joblib, pickle, torch.save/load — in more depth).

Say "next" when you're ready.